In [1]:
from pathlib import Path
import math
import requests
from io import StringIO

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Image, display
import os
import pathlib
import dataretrieval.waterdata as waterdata
from datetime import datetime, timedelta, timezone

os.environ['API_USGS_PAT'] = 'lpK2Fw6n4cZpD28LUAxmvwhluFrhyyohhlTPbV7m'

DATA_DIR   = pathlib.Path().cwd().parent / 'data'   # adjust if running from a different directory
SWORD_GPKG = pathlib.Path().cwd().parent / 'data'   # na_sword_nodes_v17b_pnw.gpkg lives alongside the notebook


In [10]:
def retrieve_stations(data_dir=Path('/Users/masa6503/repos/swot-precip-validation/data'), file_name="dfms_v1_us_station_matches.parquet", maximum_match_distance=200):
    stations = pd.read_parquet(data_dir / file_name)
    stations = stations.query(f"match_dist_m<{maximum_match_distance}")
    stations = stations[stations['usgs_site_no'].astype(str).str.len() == 8]
    return stations


def retrieve_active_stations(usgs_id_list, timeseries_start="2023-04-01T00:00:00Z"):
    tomorrow = datetime.now(tz=timezone.utc) + timedelta(days=2)
    # Format as ISO 8601 UTC midnight
    tomorrow_str = tomorrow.strftime("%Y-%m-%dT00:00:00Z")
    timeseries = timeseries_start+"/"+tomorrow_str
    all_active_ids = []
    for i in range(0, len(usgs_id_list), 50):
        end_index = min(i + 50, len(usgs_id_list))
        # print(i, end_index)
        active_ids = list(waterdata.get_latest_continuous(
            monitoring_location_id=list(usgs_id_list[i:end_index]), 
            parameter_code="00065", 
            time=timeseries
        )[0]['monitoring_location_id'].unique())
        all_active_ids.extend(active_ids)
    return (all_active_ids)

def retrieve_monitoring_locations(active_ids):
    for i in range(0, len(active_ids), 50):
        end_index = min(i + 50, len(active_ids))
        monitoring_location = waterdata.get_monitoring_locations(
            properties=['monitoring_location_name','geometry'],
            monitoring_location_id=list(active_ids[i:end_index])
        )
        if i==0:
            all_active_stations = monitoring_location[0]
        else:
            all_active_stations = pd.concat([all_active_stations, monitoring_location[0]])
    return all_active_stations

def retrieve_active_station_data(active_station_list, timeseries="2023-04-01T00:00:00Z/2026-02-01T12:31:12Z", 
                                 output_dir = '/Users/masa6503/repos/swot-precip-validation/data/usgs_stage_data'):
    filenames=[]
    for i in range(0, len(active_station_list), 50):
        end_index = min(i + 50, len(active_station_list))
        filename_to_save=f'{output_dir}/usgs_stage_data_{i}_{end_index}.parquet'
        if os.path.exists(filename_to_save):
            print(f"File {filename_to_save} already exists. Skipping retrieval.")
            filenames.append(filename_to_save)
            continue
        print(i, end_index)
        try:
            data = waterdata.get_continuous(
                monitoring_location_id=list(active_station_list[i:end_index]), 
                parameter_code="00065", 
                time=timeseries
            )[0]
            data.to_parquet(filename_to_save)
            filenames.extend(filename_to_save)
        except Exception as e:
            print(f"Error retrieving data for stations {active_station_list[i:end_index]}: {e}")
    return filenames

In [3]:
for i in range(0, 10, 50):
    print(i, min(i + 50, 10))

0 10


In [5]:
DATA_DIR

WindowsPath('c:/Users/matth/repos/swot-precip-validation/data')

In [6]:
stations = retrieve_stations(data_dir=DATA_DIR)
active_ids = retrieve_active_stations(list('USGS-'+stations['usgs_site_no']))

In [11]:
stations = retrieve_monitoring_locations(active_ids)

In [13]:
# Load built-in world dataset
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))

# Select North America
north_america = world[world["continent"] == "North America"]

# Make sure CRS matches
stations = stations.to_crs(north_america.crs)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
north_america.plot(ax=ax, color="lightgray", edgecolor="black")
stations.plot(ax=ax)

plt.show()

AttributeError: The geopandas.dataset has been deprecated and was removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.

In [8]:
len(stations)

1848

In [146]:
filenames = retrieve_active_station_data(active_ids)

File /Users/masa6503/repos/swot-precip-validation/data/usgs_stage_data/usgs_stage_data_0_50.parquet already exists. Skipping retrieval.
File /Users/masa6503/repos/swot-precip-validation/data/usgs_stage_data/usgs_stage_data_50_100.parquet already exists. Skipping retrieval.
File /Users/masa6503/repos/swot-precip-validation/data/usgs_stage_data/usgs_stage_data_100_150.parquet already exists. Skipping retrieval.
File /Users/masa6503/repos/swot-precip-validation/data/usgs_stage_data/usgs_stage_data_150_200.parquet already exists. Skipping retrieval.
File /Users/masa6503/repos/swot-precip-validation/data/usgs_stage_data/usgs_stage_data_200_250.parquet already exists. Skipping retrieval.
File /Users/masa6503/repos/swot-precip-validation/data/usgs_stage_data/usgs_stage_data_250_300.parquet already exists. Skipping retrieval.
File /Users/masa6503/repos/swot-precip-validation/data/usgs_stage_data/usgs_stage_data_300_350.parquet already exists. Skipping retrieval.
File /Users/masa6503/repos/swot

In [56]:
Path('/Users/masa6503/repos/swot-precip-validation/data')

PosixPath('/Users/masa6503/repos/swot-precip-validation/data')

In [3]:
usgs_codes = "USGS-"+np.array(stations.usgs_site_no)

In [5]:
locations = waterdata.get_monitoring_locations(monitoring_location_id=list(usgs_codes[0:100]), properties=['monitoring_location_name'])

In [92]:
import os
os.environ['API_USGS_PAT'] = 'lpK2Fw6n4cZpD28LUAxmvwhluFrhyyohhlTPbV7m'
timeseries = "2023-04-01T00:00:00Z/2026-03-30T12:31:12Z"
active_ids = waterdata.get_latest_continuous(monitoring_location_id=list(usgs_codes[100:150]), parameter_code="00065",
                                       time=timeseries)[0]['monitoring_location_id'].unique()

In [93]:
active_ids

array(['USGS-01545500', 'USGS-01618000', 'USGS-01578310', 'USGS-01553025',
       'USGS-01638500', 'USGS-02024752', 'USGS-02025500', 'USGS-02026000',
       'USGS-01559000', 'USGS-01562000', 'USGS-01570500', 'USGS-01668000',
       'USGS-01545800', 'USGS-01551500', 'USGS-01567000', 'USGS-02024750',
       'USGS-02029000', 'USGS-01636500'], dtype=object)

In [81]:
chunks = np.array_split(usgs_codes, 6)

In [87]:
chunks[1]

array(['USGS-02341460', 'USGS-02341500', 'USGS-02341505', 'USGS-02341508',
       'USGS-02342110', 'USGS-02342880', 'USGS-02342881', 'USGS-02342940',
       'USGS-02343241', 'USGS-02343260', 'USGS-02343500', 'USGS-02343801',
       'USGS-02343805', 'USGS-02344000', 'USGS-02344040', 'USGS-02347500',
       'USGS-02348060', 'USGS-02348350', 'USGS-02349500', 'USGS-02349501',
       'USGS-02349605', 'USGS-02349800', 'USGS-02350000', 'USGS-02350001',
       'USGS-02350405', 'USGS-02350511', 'USGS-02352650', 'USGS-02352700',
       'USGS-02352790', 'USGS-02352795', 'USGS-02352805', 'USGS-02352939',
       'USGS-02352945', 'USGS-02353000', 'USGS-02353075', 'USGS-02353080',
       'USGS-02355660', 'USGS-02355662', 'USGS-02355670', 'USGS-02355685',
       'USGS-02358000', 'USGS-02358634', 'USGS-02358664', 'USGS-02358684',
       'USGS-02359051', 'USGS-02359098', 'USGS-02359101', 'USGS-02359103',
       'USGS-02359170', 'USGS-02359223', 'USGS-02361500', 'USGS-02364200',
       'USGS-02365000', '

In [47]:
import os
os.environ['API_USGS_PAT'] = 'lpK2Fw6n4cZpD28LUAxmvwhluFrhyyohhlTPbV7m'
new_elevation_data = waterdata.get_continuous(
    monitoring_location_id=list(active_ids['monitoring_location_id']),
    parameter_code="00065",
    time=timeseries
)

In [123]:
new_elevation_data[0].to_parquet('/Users/masa6503/repos/swot-precip-validation/data/test.parquet')
test_read = pd.read_parquet('/Users/masa6503/repos/swot-precip-validation/data/test.parquet')

In [124]:
test_read

,time_series_id,monitoring_location_id,parameter_code,statistic_id,time,value,unit_of_measure,approval_status,qualifier,last_modified,continuous_id
0,d8451cd6afc7492db286902034eddc69,USGS-01545500,00065,00011,2023-04-01 00:00:00+00:00,3.24,ft,Approved,None,2025-09-04 01:50:53.346384+00:00,05c18956-c83e-44f0-81ec-22c799c81c7e
1,96c780f8e164436cbf092126f2976c40,USGS-01545800,00065,00011,2023-04-01 00:00:00+00:00,9.48,ft,Approved,None,2025-09-03 21:54:54.947671+00:00,32df4ae5-a295-44c9-bb7d-63a4f8a5aa67
2,05c4f0f4aee340e887f26426eceb365f,USGS-01551500,00065,00011,2023-04-01 00:00:00+00:00,4.52,ft,Approved,None,2025-09-03 13:15:18.371829+00:00,65346b16-59b1-49fd-8e4e-710af2f368d3
3,2517d8e283aa42ba83f28c3e37e07eff,USGS-01553025,00065,00011,2023-04-01 00:00:00+00:00,10.33,ft,Approved,None,2025-09-03 14:58:37.408115+00:00,4b6bbb40-354a-4281-a0b9-d618d3ba4ede
4,8f38ec92c62f4d59bb7cba148dbd9350,USGS-01559000,00065,00011,2023-04-01 00:00:00+00:00,2.70,ft,Approved,None,2025-08-06 17:18:16.275327+00:00,b1efdefa-e94b-48b1-a8bf-69e52238886c
...,...,...,...,...,...,...,...,...,...,...,...
1944839,04abb4817ccc4bb089e928ca8a3b40be,USGS-01636500,00065,00011,2026-03-03 22:15:00+00:00,2.90,ft,Provisional,None,2026-03-03 22:36:24.005858+00:00,f735a1a9-3c68-4bc9-875d-1d44dab763bc
1944840,03d73351a85949b18c758a4ffd0ff670,USGS-02024750,00065,00011,2026-03-03 22:15:00+00:00,5.76,ft,Provisional,None,2026-03-03 22:25:21.809978+00:00,11d80f51-1a47-4eb0-a84e-64f6c4be3e84
1944841,7a5cdb4ed7bc465686474f926ec08899,USGS-02029000,00065,00011,2026-03-03 22:15:00+00:00,4.48,ft,Provisional,None,2026-03-03 22:23:26.939186+00:00,ad4be2f9-fcf7-40eb-b2e5-1e7aaba35c82
1944842,d8451cd6afc7492db286902034eddc69,USGS-01545500,00065,00011,2026-03-03 22:30:00+00:00,2.46,ft,Provisional,None,2026-03-03 22:39:15.885267+00:00,b2646dc4-1df5-4a78-9a46-8acfa54eb53d


In [ ]:
def fetch_usgs_gage_height_iv(site_no: str, start: str, end: str) -> pd.DataFrame:
    """
    Fetch USGS instantaneous gage height (parameter 00065, ft) from NWIS.
    Returns raw 5-minute readings with columns: time (tz-naive UTC), gage_height_m.
    """
    params = {
        "format":      "json",
        "sites":       site_no,
        "startDT":     start,
        "endDT":       end,
        "parameterCd": "00065"
    }
    resp = requests.get(
        "https://nwis.waterservices.usgs.gov/nwis/iv/",
        params=params, timeout=120,
    )
    resp.raise_for_status()

    ts_list = resp.json()["value"]["timeSeries"]
    if not ts_list:
        return pd.DataFrame(columns=["time", "gage_height_m"])

    records = ts_list[0]["values"][0]["value"]
    df = pd.DataFrame(records)
    # dateTime comes back with local offset; convert to UTC and drop tzinfo
    df["time"] = pd.to_datetime(df["dateTime"], utc=True).dt.tz_convert(None)
    df["gage_height_m"] = pd.to_numeric(df["value"], errors="coerce") * 0.3048
    return df[["time", "gage_height_m"]].dropna().reset_index(drop=True)


# ── Fetch USGS IV only for the date span covered by SWOT passes ──────────
swot_start = combined['nodetime'].min().strftime("%Y-%m-%d")
swot_end   = combined['nodetime'].max().strftime("%Y-%m-%d")
# print(f"SWOT pass window: {swot_start} → {swot_end}  ({len(swot_filter_df)} passes)")
print("Fetching USGS IV gage height for that window …")

usgs_iv = fetch_usgs_gage_height_iv("14174000", swot_start, swot_end)